# Wayfinder Eval Pipeline v3 — Multi-Lens Geometry

Rebuilds the proof network with **typed anchors** (6 lens categories) and
runs the full eval suite to measure multi-lens coherence scoring:

1. **Rebuild** — Re-extract entities with typed anchors, rebuild DB with category + confidence + per-category IDF
2. **Eval Retrieval** (EXP-2.1): Nav vs Dense recall with multi-lens coherence scoring
3. **Eval Spreading** (EXP-2.2): Spreading activation benefit
4. **Anchor Gap Analysis** (EXP-0.2): Category-aware gap analysis on rebuilt DB

**What changed:** Anchors now carry `{label, category, confidence}`. Six lens categories:
`semantic`, `structural`, `lexical`, `locality`, `proof`, `constant`.
Scoring uses geometric mean across populated lenses (multi-lens coherence).
`general` anchors → confidence 0.0. `.lake/packages` paths → confidence 0.05.

**Setup:** Runtime → Change runtime type → T4 GPU

In [ ]:
# Clone repo and install deps
!git clone https://github.com/rohanvinaik/Wayfinder.git /content/wayfinder
%cd /content/wayfinder
!pip install -q torch numpy pyyaml sentence-transformers transformers

In [ ]:
# Verify GPU available
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected. Runtime → Change runtime type → T4 GPU')

In [ ]:
# Mount Google Drive and copy source data
from google.colab import drive
import os
import shutil

drive.mount('/content/drive')

drive_base = '/content/drive/MyDrive/Wayfinder'  # <-- EDIT this path

# Source data for DB rebuild (required)
source_files = {
    f'{drive_base}/leandojo_mathlib.jsonl': 'data/leandojo_mathlib.jsonl',
    f'{drive_base}/corpus.jsonl': 'data/leandojo_benchmark_4/corpus.jsonl',
}

# Pre-built artifacts (copy if available, will rebuild if missing)
artifact_files = {
    f'{drive_base}/nav_eval.jsonl': 'data/nav_eval.jsonl',
    f'{drive_base}/nav_train.jsonl': 'data/nav_train.jsonl',
    f'{drive_base}/NAV-002_step5000.pt': 'models/NAV-002_step5000.pt',
}

print('=== Source data (for DB rebuild) ===')
for src, dst in source_files.items():
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if os.path.exists(src):
        if not os.path.exists(dst):
            print(f'  Copying {os.path.basename(src)} ({os.path.getsize(src)/1e6:.0f} MB)...')
            shutil.copy2(src, dst)
        else:
            print(f'  ✓ {dst} already exists')
    else:
        print(f'  ✗ {src} NOT FOUND — needed for DB rebuild')

print('\n=== Artifacts ===')
for src, dst in artifact_files.items():
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if os.path.exists(src):
        if not os.path.exists(dst):
            print(f'  Copying {os.path.basename(src)} ({os.path.getsize(src)/1e6:.0f} MB)...')
            shutil.copy2(src, dst)
        else:
            print(f'  ✓ {dst} already exists')
    else:
        print(f'  ✗ {os.path.basename(src)} not on Drive (optional)')

---
## Step 1: Rebuild Proof Network with Typed Anchors

Re-extracts entities with multi-lens anchor categories and rebuilds the DB.
This is the critical step — the v2 DB had flat anchors; v3 has typed anchors
with `{label, category, confidence}` across 6 lens categories.

**Time:** ~5-10 min on Colab (extraction + DB build + IDF)

In [ ]:
import os
import time

os.chdir('/content/wayfinder')
os.environ['PYTHONPATH'] = '/content/wayfinder'

# Check source data exists
assert os.path.exists('data/leandojo_mathlib.jsonl'), \
    'Missing data/leandojo_mathlib.jsonl — upload from Drive'
assert os.path.exists('data/leandojo_benchmark_4/corpus.jsonl'), \
    'Missing data/leandojo_benchmark_4/corpus.jsonl — upload from Drive'

# Remove old entities and DB to force full rebuild
for old in ['data/proof_network_entities.jsonl', 'data/proof_network.db']:
    if os.path.exists(old):
        os.remove(old)
        print(f'  Removed old {old}')

print('\n=== Step 1a: Extract entities with typed anchors ===')
t0 = time.time()
!PYTHONPATH=/content/wayfinder python scripts/extract_proof_network.py \
    --input data/leandojo_mathlib.jsonl \
    --output data/proof_network_entities.jsonl \
    --corpus data/leandojo_benchmark_4/corpus.jsonl
t1 = time.time()
print(f'\nExtraction time: {t1-t0:.1f}s')

# Verify typed anchors in output
import json
with open('data/proof_network_entities.jsonl') as f:
    first = json.loads(f.readline())
if isinstance(first['anchors'][0], dict):
    cats = {a['category'] for a in first['anchors']}
    print(f'\n✓ Typed anchors confirmed. Categories in first entity: {sorted(cats)}')
else:
    print('\n✗ WARNING: Anchors are plain strings — code may not have the multi-lens changes')

In [ ]:
import time

print('=== Step 1b: Build proof_network.db with categories + confidences ===')
t0 = time.time()
!PYTHONPATH=/content/wayfinder python -m scripts.build_proof_network_db \
    --input data/proof_network_entities.jsonl \
    --output data/proof_network.db
t1 = time.time()
print(f'\nDB build time: {t1-t0:.1f}s')

In [ ]:
# Verify DB schema has categories and per-category IDF
import sqlite3

conn = sqlite3.connect('data/proof_network.db')

n_entities = conn.execute('SELECT COUNT(*) FROM entities').fetchone()[0]
n_anchors = conn.execute('SELECT COUNT(*) FROM anchors').fetchone()[0]
n_traced = conn.execute("SELECT COUNT(*) FROM entities WHERE provenance='traced'").fetchone()[0]
n_premise = conn.execute("SELECT COUNT(*) FROM entities WHERE provenance='premise_only'").fetchone()[0]

print(f'Entities: {n_entities:,} ({n_traced:,} traced, {n_premise:,} premise_only)')
print(f'Anchors: {n_anchors:,}')

# Anchor category distribution
cats = conn.execute(
    'SELECT category, COUNT(*) FROM anchors GROUP BY category ORDER BY COUNT(*) DESC'
).fetchall()
print('\nAnchor categories:')
for cat, count in cats:
    print(f'  {cat}: {count:,}')

# Per-category IDF table
try:
    cat_idf_count = conn.execute('SELECT COUNT(*) FROM anchor_category_idf').fetchone()[0]
    print(f'\nPer-category IDF entries: {cat_idf_count:,}')
except Exception:
    print('\n✗ anchor_category_idf table not found — IDF may be global only')

# Confidence distribution
confs = conn.execute(
    'SELECT ROUND(confidence, 1) as c, COUNT(*) FROM entity_anchors GROUP BY c ORDER BY c'
).fetchall()
print('\nAnchor confidence distribution:')
for conf, count in confs:
    print(f'  {conf}: {count:,}')

conn.close()

---
## Step 2: Eval Retrieval (EXP-2.1)

Compares navigational retrieval vs dense cosine similarity baseline.
Now uses **multi-lens coherence scoring**: per-category anchor overlap with
geometric mean across populated lenses.

**Gate:** nav recall@16 ≥ 80% of dense recall@16

In [ ]:
import os

os.chdir('/content/wayfinder')
os.environ['PYTHONPATH'] = '/content/wayfinder'

!PYTHONPATH=/content/wayfinder python scripts/eval_retrieval.py \
    --config configs/wayfinder.yaml \
    --checkpoint models/NAV-002_step5000.pt \
    --device cuda \
    --samples 500 \
    --output runs/eval_retrieval_v3_gpu.json

In [ ]:
import json
import os

result_path = 'runs/eval_retrieval_v3_gpu.json'
if not os.path.exists(result_path):
    print(f'ERROR: {result_path} not found. Check the eval_retrieval output above for errors.')
else:
    with open(result_path) as f:
        report = json.load(f)

    print('=== Universe Coverage ===')
    cov = report['universe_coverage']
    print(f"  Mean: {cov['mean']:.1%}")
    print(f"  Fully covered: {cov['fully_covered']}/{report['samples']}")
    print(f"  Zero covered: {cov['zero_covered']}/{report['samples']}")

    print('\n=== Raw Retrieval ===')
    for k in [1, 4, 8, 16]:
        nr = report['nav_retrieval'][f'recall@{k}']
        dr = report['dense_retrieval'][f'recall@{k}']
        print(f"  recall@{k}: nav={nr:.4f}  dense={dr:.4f}  delta={nr-dr:+.4f}")

    print('\n=== Conditional Retrieval (reachable premises only) ===')
    for k in [1, 4, 8, 16]:
        nr = report['nav_conditional_retrieval'][f'cond_recall@{k}']
        dr = report['dense_conditional_retrieval'][f'cond_recall@{k}']
        print(f"  cond_recall@{k}: nav={nr:.4f}  dense={dr:.4f}  delta={nr-dr:+.4f}")

    print('\n=== Timing ===')
    t = report['timing']
    print(f"  Nav: {t['nav_avg_ms']:.1f}ms  Dense: {t['dense_avg_ms']:.1f}ms  Speedup: {t['speedup']:.1f}x")

---
## Step 3: Eval Spreading (EXP-2.2)

Tests whether spreading activation from proof history improves premise retrieval.

**Gate:** ≥5% recall@16 improvement at proof step 3+

In [ ]:
!PYTHONPATH=/content/wayfinder python scripts/eval_spreading.py \
    --config configs/wayfinder.yaml \
    --checkpoint models/NAV-002_step5000.pt \
    --device cuda \
    --samples 500 \
    --output runs/eval_spreading_v3_gpu.json

In [ ]:
import json
import os

result_path = 'runs/eval_spreading_v3_gpu.json'
if not os.path.exists(result_path):
    print(f'ERROR: {result_path} not found. Check the eval_spreading output above for errors.')
else:
    with open(result_path) as f:
        spread_report = json.load(f)
    print(json.dumps(spread_report, indent=2))

---
## Step 4: Anchor Gap Analysis (EXP-0.2)

Category-aware gap analysis on the rebuilt DB with typed anchors.
Now reports per-lens gap distribution and excludes trivial anchors
(`general`, `.lake/packages` paths) from gap surfacing.

**Gate:** recall@16 ≥ 70% on perfect queries

In [ ]:
!PYTHONPATH=/content/wayfinder python scripts/anchor_gap_analysis.py \
    --db data/proof_network.db \
    --samples 500 \
    --output runs/anchor_gap_v3.jsonl

---
## Step 5: Save rebuilt DB to Drive + download results

In [ ]:
import os
import shutil

# Save rebuilt DB back to Drive for future runs
drive_base = '/content/drive/MyDrive/Wayfinder'
if os.path.exists(drive_base):
    for src, label in [
        ('data/proof_network.db', 'proof_network_v3.db'),
        ('data/proof_network_entities.jsonl', 'proof_network_entities_v3.jsonl'),
    ]:
        if os.path.exists(src):
            dst = f'{drive_base}/{label}'
            print(f'  Saving {label} ({os.path.getsize(src)/1e6:.0f} MB) to Drive...')
            shutil.copy2(src, dst)
    print('  Done.')
else:
    print(f'  Drive path {drive_base} not found — skipping save')

# Collect result files for download
result_files = [
    'runs/eval_retrieval_v3_gpu.json',
    'runs/eval_spreading_v3_gpu.json',
    'runs/anchor_gap_v3.jsonl',
]

os.makedirs('/content/results', exist_ok=True)
for f in result_files:
    if os.path.exists(f):
        shutil.copy(f, f'/content/results/{os.path.basename(f)}')
        print(f'  ✓ {f}')
    else:
        print(f'  ✗ {f} (not found)')

shutil.make_archive('/content/wayfinder_eval_results_v3', 'zip', '/content/results')
print('\nDownload: /content/wayfinder_eval_results_v3.zip')

try:
    from google.colab import files
    files.download('/content/wayfinder_eval_results_v3.zip')
except ImportError:
    print('Not in Colab — download manually from the file browser')